# Tuning & Experiment Tracking — Notebook

**Week 8 · CS 203 · Software Tools and Techniques for AI**

Prof. Nipun Batra · IIT Gandhinagar

This notebook covers:
1. Grid Search vs Random Search
2. Bayesian Optimization with Optuna
3. Nested Cross-Validation
4. PyTorch reproducibility (seeds, deterministic mode, DataLoader)
5. Multi-seed reporting
6. W&B experiment tracking

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy matplotlib scikit-learn optuna torch wandb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, StratifiedKFold
)
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")

# Generate a reproducible dataset
np.random.seed(42)
X = np.random.rand(1000, 8)
y = (X[:, 0] * 2 + X[:, 2] - X[:, 5] + np.random.randn(1000) * 0.3 > 1).astype(int)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance: {y.mean():.1%} positive")

## 1. Grid Search vs Random Search

In [ ]:
# Grid Search: try every combination
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid.fit(X, y)

n_grid = len(grid.cv_results_['params'])
print(f"Grid Search: {n_grid} combinations evaluated")
print(f"Best score:  {grid.best_score_:.4f}")
print(f"Best params: {grid.best_params_}")

In [ ]:
# Random Search: same budget, sample randomly
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features': uniform(0.1, 0.9),
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    n_iter=n_grid,  # same budget as grid
    cv=5, scoring='accuracy', random_state=42, n_jobs=-1
)
random_search.fit(X, y)

print(f"Random Search: {n_grid} combinations evaluated")
print(f"Best score:    {random_search.best_score_:.4f}")
print(f"Best params:   {random_search.best_params_}")
print(f"\nRandom {'wins' if random_search.best_score_ > grid.best_score_ else 'loses'} "
      f"by {abs(random_search.best_score_ - grid.best_score_):.4f}")

## 2. Bayesian Optimization with Optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X, y, cv=5, n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=n_grid)  # same budget

print(f"Optuna: {n_grid} trials")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Compare all three
print("=== Same budget comparison ===")
print(f"Grid Search:   {grid.best_score_:.4f}")
print(f"Random Search: {random_search.best_score_:.4f}")
print(f"Optuna:        {study.best_value:.4f}")

In [ ]:
# Optuna visualization: optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [ ]:
# Optuna visualization: parameter importances
fig = optuna.visualization.plot_param_importances(study)
fig.show()

## 3. Nested Cross-Validation

`GridSearchCV.best_score_` is optimistic due to selection bias. Nested CV gives an unbiased estimate.

In [ ]:
# Inner loop: tune hyperparameters
inner_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={'max_depth': [5, 10, 15], 'n_estimators': [100, 200]},
    cv=3
)

# Outer loop: evaluate the tuned model
outer_scores = cross_val_score(inner_cv, X, y, cv=5)

# Compare
inner_cv.fit(X, y)
print(f"GridSearchCV.best_score_ (optimistic): {inner_cv.best_score_:.4f}")
print(f"Nested CV score (unbiased):            {outer_scores.mean():.4f} +/- {outer_scores.std():.4f}")
print(f"\nThe gap ({inner_cv.best_score_ - outer_scores.mean():.4f}) is the optimistic bias.")

## 4. PyTorch Reproducibility

In [ ]:
import torch
import torch.nn as nn
import random
import os

def set_seed_full(seed=42):
    """Complete seed function for PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Without seeds: different every time
print("=== Without seeds ===")
for i in range(3):
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y_pred = model(x)
    print(f"  Run {i+1}: first output = {y_pred[0].item():.6f}")

# With seeds: identical every time
print("\n=== With seeds ===")
for i in range(3):
    set_seed_full(42)
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y_pred = model(x)
    print(f"  Run {i+1}: first output = {y_pred[0].item():.6f}")

In [ ]:
# DataLoader reproducibility
from torch.utils.data import DataLoader, TensorDataset

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print("=== DataLoader with worker seeding ===")
for i in range(3):
    set_seed_full(42)
    g = torch.Generator()
    g.manual_seed(42)
    dataset = TensorDataset(torch.randn(100, 10), torch.randint(0, 2, (100,)))
    loader = DataLoader(
        dataset, batch_size=16, shuffle=True,
        worker_init_fn=seed_worker, generator=g
    )
    first_batch = next(iter(loader))
    print(f"  Run {i+1}: first batch sum = {first_batch[0].sum().item():.4f}, "
          f"labels = {first_batch[1][:5].tolist()}")

## 5. Multi-Seed Reporting

Better than one deterministic run: report mean +/- std across seeds.

In [ ]:
results = []
for seed in [42, 123, 456, 789, 1024]:
    np.random.seed(seed)
    X_s = np.random.rand(500, 8)
    y_s = (X_s[:, 0] * 2 + X_s[:, 2] - X_s[:, 5] + np.random.randn(500) * 0.3 > 1).astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    results.append(acc)
    print(f"  Seed {seed:>4d}: {acc:.4f}")

print(f"\nAccuracy: {np.mean(results):.4f} +/- {np.std(results):.4f}")
print("This is more informative than a single deterministic run!")

## 6. W&B Integration (Optional)

Uncomment and run if you have a W&B account.

In [ ]:
# import wandb

# wandb.init(project="movie-predictor-demo", config={
#     "n_estimators": 100,
#     "max_depth": 10,
#     "seed": 42,
# })

# model = RandomForestClassifier(
#     n_estimators=wandb.config.n_estimators,
#     max_depth=wandb.config.max_depth,
#     random_state=wandb.config.seed,
# )

# np.random.seed(42)
# X_w = np.random.rand(1000, 8)
# y_w = (X_w[:, 0] * 2 + X_w[:, 2] - X_w[:, 5] + np.random.randn(1000) * 0.3 > 1).astype(int)
# X_train, X_test, y_train, y_test = train_test_split(X_w, y_w, test_size=0.2, random_state=42)

# model.fit(X_train, y_train)
# train_acc = model.score(X_train, y_train)
# test_acc = model.score(X_test, y_test)

# wandb.log({"train_accuracy": train_acc, "test_accuracy": test_acc})

# for i, imp in enumerate(model.feature_importances_):
#     wandb.log({f"feature_{i}_importance": imp})

# print(f"Train: {train_acc:.4f}, Test: {test_acc:.4f}")
# print(f"View at: {wandb.run.url}")
# wandb.finish()

## Summary

**Key takeaways:**
- Grid search tries all combos; random search samples; Optuna learns from past trials
- Nested CV gives unbiased estimates of tuned model performance
- PyTorch reproducibility requires seeds + deterministic algorithms + DataLoader seeding
- Report results over multiple seeds for reliability
- W&B automates experiment tracking (params, metrics, code, artifacts)
- MLflow is a self-hosted alternative